# Incremental Ingestion

Incremental ingestion loads only the files that arrived since the last run, so a pipeline does not read the same data twice. This module covers three patterns:

- `COPY INTO`: idempotent, file-level batch loading into Delta tables.
- Auto Loader (`cloudFiles`): scalable streaming file ingestion with schema inference.
- Structured Streaming: trigger modes, output modes, checkpoints and schema rescue.

Advanced streaming patterns (stream-static joins, watermarking, `badRecordsPath`, Change Data Feed) are in <a href="$./BONUS_04_streaming_advanced">BONUS_04 — Streaming Advanced</a>.

## Learning Objectives

After completing this module you will be able to:

- Use `COPY INTO` to load files idempotently into a Delta table.
- Explain why `COPY INTO` is safe to re-run (file tracking) and how to force a reload.
- Configure Auto Loader (`cloudFiles`) with schema inference and evolution.
- Choose an Auto Loader file detection mode: directory listing or file notification (file events).
- Run incremental streaming queries with `trigger(availableNow=True)` and explain when `processingTime` triggers apply (classic compute only).
- Handle schema changes with rescue mode (`_rescued_data`), and malformed records with the parser mode or `badRecordsPath`.

| Exam Domain | Weight |
|---|---|
| 2 — Data Ingestion and Loading | **21%** |

## Setup

The setup notebook checks your training catalog and exports its variables, such as `CATALOG` and `DATASET_PATH`. The cells after it import libraries and prepare the simulated source files for all ingestion demos in this module.



In [0]:
%run ../../setup/00_setup

### Configuration

Every demo needs a folder for source files, one for checkpoints and one for schema locations. The cells below define these paths and remove files from an earlier run. Then they split the source data into parts that arrive one by one, like daily deliveries to a landing folder. Only Day 1 and Batch 1 are written now, and the other parts stay in memory until a demo writes them.

In [0]:
# Libraries used by the demos
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import time

In [0]:
# [1/2] Define catalog context and all demo paths
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

# Source data (real dataset)
SOURCE_CUSTOMERS = f"{DATASET_PATH}/customers/customers.csv"
SOURCE_ORDERS = f"{DATASET_PATH}/orders/orders_batch.json"

# Demo paths (simulated file arrivals)
DEMO_BASE_PATH = f"{DATASET_PATH}/ingestion_demo"
BATCH_SOURCE_PATH = f"{DEMO_BASE_PATH}/batch_source"
STREAM_SOURCE_PATH = f"{DEMO_BASE_PATH}/stream_source"

# Technical paths
CHECKPOINT_BASE_PATH = f"{DEMO_BASE_PATH}/checkpoints"
SCHEMA_BASE_PATH = f"{DEMO_BASE_PATH}/schemas"

print(f"Paths defined. Base: {DEMO_BASE_PATH}")

In [0]:
# [2/2] Remove files from a previous run, so every run starts from the same state
dbutils.fs.rm(DEMO_BASE_PATH, True)
print(f"Demo environment prepared at: {DEMO_BASE_PATH}")

In [0]:
# Show the setup variables and the source paths used below
display(
    spark.createDataFrame([
        ("CATALOG", CATALOG),
        ("BRONZE_SCHEMA", BRONZE_SCHEMA),
        ("SILVER_SCHEMA", SILVER_SCHEMA),
        ("USER", raw_user),
        ("CUSTOMERS_CSV", SOURCE_CUSTOMERS),
        ("STREAMING_SOURCE_PATH", STREAM_SOURCE_PATH)
    ], ["Variable", "Value"])
)

In [0]:
# [1/2] Split the customers CSV into 4 parts that play the role of daily files
df_customers = spark.read.option("header", "true").csv(SOURCE_CUSTOMERS)
df_batch_day1, df_batch_day2, df_batch_day3, df_batch_day4 = df_customers.randomSplit([0.25]*4, seed=42)

# Write Day 1 now; days 2-4 "arrive" later in the demo
df_batch_day1.write.mode("overwrite").option("header", "true").csv(f"{BATCH_SOURCE_PATH}/day1")
print(f"Batch Data: Day 1 ready at {BATCH_SOURCE_PATH}/day1")
print(f"Day 2-4 in memory — will be saved on demand to simulate daily arrivals")

In [0]:
# [2/2] Split the orders JSON into 20 parts that arrive as separate batches
SOURCE_STREAM_FILES = f"{DATASET_PATH}/orders/stream/*.json"
df_all_orders = spark.read.json(SOURCE_STREAM_FILES)

# 20 parts of 5% each simulate files that keep arriving
stream_batches = df_all_orders.randomSplit([0.05] * 20, seed=42)

# Write Batch 1 now, so the stream has a file to start with
stream_batches[0].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/batch_01")
print(f"Stream Data: Batch 1 ready at {STREAM_SOURCE_PATH}/batch_01")

### Data Loading Methods Overview

| Feature | CTAS | COPY INTO | Auto Loader |
|---------|------|-----------|-------------|
| **Incremental** | No | Yes | Yes |
| **Idempotent** | No | Yes | Yes |
| **Schema Evolution** | No | Limited | Advanced |
| **File Tracking** | No | Metadata | Checkpoint |
| **Scalability** | Low | Medium | High |
| **Streaming** | No | No | Yes |
| **Use Case** | One-time | Scheduled batch | Real-time/Streaming |

> **Exam Tip:** Auto Loader (`cloudFiles`) is the **recommended** ingestion method for new projects. `COPY INTO` fits scheduled batch loads of up to thousands of files.



## COPY INTO: Batch Loading

> **Summary** (see [COPY INTO](https://learn.microsoft.com/azure/databricks/sql/language-manual/delta-copy-into)): `COPY INTO` loads data from a file location into a Delta table. It is idempotent: files that were already loaded are skipped on the next run. Use it for batch loads of up to thousands of files. For millions of files or continuous ingestion, use Auto Loader.

A typical use is a daily or hourly job that loads the files a source system drops into cloud storage. `COPY INTO` tracks the loaded files in the table metadata, so the job can run again and does not load duplicates.

Syntax:
```sql
COPY INTO target_table
FROM (SELECT ... FROM 'path/')
FILEFORMAT = { CSV | JSON | PARQUET | AVRO | ORC | TEXT | BINARYFILE }
FORMAT_OPTIONS ('key' = 'value', ...)
COPY_OPTIONS ('key' = 'value', ...)
```

| Clause | Key Options | Description |
|---|---|---|
| `FILEFORMAT` | `CSV`, `JSON`, `PARQUET`, `AVRO` | Source file format |
| `FORMAT_OPTIONS` | `header`, `inferSchema`, `delimiter` | Format-specific parsing options |
| `COPY_OPTIONS` | `mergeSchema`, `force` | `mergeSchema=true` enables schema evolution; `force=true` reloads already-processed files |

`mergeSchema` can go in both clauses and means different things: in `FORMAT_OPTIONS` it merges the schemas of the source files, in `COPY_OPTIONS` it lets `COPY INTO` evolve the schema of the target table.

Syntax reference for `COPY INTO`:

| Element | Syntax |
|---------|--------|
| Basic load | `COPY INTO table FROM '/path/' FILEFORMAT = CSV` |
| With transform | `COPY INTO table FROM (SELECT col, CAST(ts AS TIMESTAMP) FROM '/path/') FILEFORMAT = JSON` |
| Format options | `FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')` |
| Copy options | `COPY_OPTIONS ('mergeSchema' = 'true')` |
| Force re-load | `COPY INTO table FROM '/path/' FILEFORMAT = CSV COPY_OPTIONS ('force' = 'true')` (disables idempotency and reloads every file) |
| Supported formats | `CSV` \| `JSON` \| `AVRO` \| `ORC` \| `PARQUET` \| `TEXT` \| `BINARYFILE` |

### Demo: COPY INTO from CSV

Daily CSV exports from a source system are a typical input for `COPY INTO`. You create the target table first. Then `COPY INTO` loads the Day 1 files, and its `SELECT` casts `registration_date` to a date and adds `_ingestion_timestamp`.

In [0]:
# Target table for the COPY INTO demos
TABLE_CUSTOMERS = f"{BRONZE_SCHEMA}.customers_batch"

Drop the table from an earlier run, then create it with explicit column types.

In [0]:
# Drop the table from an earlier run, so the first load starts empty
spark.sql(f"DROP TABLE IF EXISTS {TABLE_CUSTOMERS}")
print(f"Dropped {TABLE_CUSTOMERS} if it existed")

In [0]:
# Create the target table with explicit column types
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_CUSTOMERS} (
  customer_id STRING,
  first_name STRING,
  last_name STRING,
  email STRING,
  phone STRING,
  city STRING,
  state STRING,
  country STRING,
  registration_date DATE,
  customer_segment STRING,
  _ingestion_timestamp TIMESTAMP
) USING DELTA
COMMENT 'Customers data - Bronze layer'
""")
print(f"Created {TABLE_CUSTOMERS}")

Run `COPY INTO` for the Day 1 folder. The first result shows how many rows were loaded, the second one shows the table.

> **Note:** about 3% of rows in the source dataset have no `customer_id` on purpose, for the data-quality exercises later. The preview lists them last. Some `customer_id` values also repeat with a different name (for example `CUST000007`: Amber George and Bruno Marquez), again on purpose for the deduplication exercises. The load is not broken and did not duplicate rows.

In [0]:
# Load Day 1 data
result = spark.sql(f"""
COPY INTO {TABLE_CUSTOMERS}
FROM (
  SELECT 
    customer_id,
    first_name,
    last_name,
    email,
    phone,
    city,
    state,
    country,
    TO_DATE(registration_date, 'yyyy-MM-dd') as registration_date,
    customer_segment,
    current_timestamp() as _ingestion_timestamp
  FROM '{BATCH_SOURCE_PATH}/day1'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

display(result)
# NULLS LAST: about 3% of the source rows have no customer_id on purpose (data-quality labs)
display(spark.sql(f"SELECT * FROM {TABLE_CUSTOMERS} ORDER BY customer_id NULLS LAST"))

### Demo: Idempotency

A scheduled job sometimes runs twice, for example after a retry. Run the same `COPY INTO` again: file tracking skips the files that were already loaded. In the last table the difference between the row counts is 0.

<img src="../../../assets/images/nb04-copy-into-idempotency.png" alt="COPY INTO remembers what it loaded: a repeated run adds no rows, a run after new files loads only those files, and force = true reloads old files as duplicates" width="800">

In [0]:
count_before = spark.table(TABLE_CUSTOMERS).count()

# Re-run on the same files: expect num_affected_rows = 0
display(spark.sql(f"""
COPY INTO {TABLE_CUSTOMERS}
FROM (
  SELECT 
    customer_id, first_name, last_name, email, phone,
    city, state, country,
    TO_DATE(registration_date, 'yyyy-MM-dd') as registration_date,
    customer_segment,
    current_timestamp() as _ingestion_timestamp
  FROM '{BATCH_SOURCE_PATH}/*'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
"""))

In [0]:
# Row count after the second run
count_after = spark.table(TABLE_CUSTOMERS).count()

In [0]:
# Before and after the second run: the difference should be 0
display(
    spark.createDataFrame([
        ("Before", count_before),
        ("After", count_after),
        ("Difference", count_after - count_before)
    ], ["State", "Count"])
)

### Demo: Adding More Days

In real work new files land in the same folder every day. The next cells write the files for day2, day3 and day4. One more `COPY INTO` loads only these new files and does not read day1 again. Compare `New rows` with the count before the run.

In [0]:
# Add Day 2 data
count_before_days = spark.table(TABLE_CUSTOMERS).count()

df_batch_day2.write.mode("overwrite").option("header", "true").csv(f"{BATCH_SOURCE_PATH}/day2")
print(f"Day 2 data written to {BATCH_SOURCE_PATH}/day2")

In [0]:
# Add Day 3 data
df_batch_day3.write.mode("overwrite").option("header", "true").csv(f"{BATCH_SOURCE_PATH}/day3")
print(f"Day 3 data written to {BATCH_SOURCE_PATH}/day3")

In [0]:
# Add Day 4 data
df_batch_day4.write.mode("overwrite").option("header", "true").csv(f"{BATCH_SOURCE_PATH}/day4")
print(f"Day 4 data written to {BATCH_SOURCE_PATH}/day4")

In [0]:
# The same COPY INTO now loads only the new files (day2, day3, day4)
result = spark.sql(f"""
COPY INTO {TABLE_CUSTOMERS}
FROM (
  SELECT 
    customer_id,
    first_name,
    last_name,
    email,
    phone,
    city,
    state,
    country,
    TO_DATE(registration_date, 'yyyy-MM-dd') as registration_date,
    customer_segment,
    current_timestamp() as _ingestion_timestamp
  FROM '{BATCH_SOURCE_PATH}/*'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
""")

count_after_days = spark.table(TABLE_CUSTOMERS).count()
print(f"Count before: {count_before_days}")
print(f"Count after:  {count_after_days}")
print(f"New rows:     {count_after_days - count_before_days}")

display(result)

## Auto Loader: Streaming Ingestion

> **Databricks documentation:** "Auto Loader incrementally and efficiently processes new data files as they arrive in cloud storage."
> — *[What is Auto Loader?](https://learn.microsoft.com/azure/databricks/ingestion/cloud-object-storage/auto-loader/)*
>
> In short: the `cloudFiles` source takes an input directory, processes each new file once and records its progress in the checkpoint. It supports schema inference and schema evolution and scales to billions of files.

Use Auto Loader when files keep arriving or when a folder holds too many files for `COPY INTO`. It keeps its progress in a checkpoint, so each file is loaded exactly once, and it can evolve the schema when new columns appear.

The `readStream` pattern:
```python
spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json|csv|parquet|...")
    .option("cloudFiles.schemaLocation", "/path/to/schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns|rescue|failOnNewColumns|none")
    .load("/source/path")
```

| Option | Values | Description |
|---|---|---|
| `cloudFiles.format` | `json`, `csv`, `parquet`, `avro` | Source file format |
| `cloudFiles.schemaLocation` | path | Where Auto Loader persists the inferred schema |
| `cloudFiles.schemaEvolutionMode` | `addNewColumns` (default when **no** schema is provided), `rescue`, `failOnNewColumns`, `none` (default when a schema is provided via `.schema(...)`) | How to handle schema changes |
| `cloudFiles.inferColumnTypes` | `true` / `false` | Infer precise column types (default: all strings) |
| `cloudFiles.includeExistingFiles` | `true` / `false` | Process files that existed before the stream started |
| `cloudFiles.maxFilesPerTrigger` | integer | Limits files processed per micro-batch |

> **Exam Tip:** Auto Loader uses the `cloudFiles` format with **checkpoint-based** exactly-once processing. It is the **recommended** method for new ingestion pipelines.



### Trigger Modes & Output Modes

The trigger decides when a stream processes data and whether it stops. The output mode decides which rows each micro-batch writes to the target.

| Trigger Mode | Behavior | Use Case |
|------|----------|----------|
| `availableNow=True` | Process all available → stop | Scheduled jobs (works on serverless and classic) |
| `processingTime="10 seconds"` | Micro-batch every N seconds, runs until stopped | Low-latency continuous jobs, **classic compute only** (not supported on serverless) |
| `once=True` | One batch, then stop (deprecated) | Use `availableNow` instead |
| `realTime="5 minutes"` | Real-time mode: records processed as they arrive in long-running batches | Sub-second operational workloads, **classic compute only**, outside the exam guide |
| *(no trigger)* | Same as `processingTime="0 seconds"` | On serverless the query fails with `INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED`: always set a trigger |

<img src="../../../assets/images/nb04-streaming-triggers.png" alt="Streaming triggers on one time axis: availableNow (serverless and classic), processingTime (classic only), no trigger (error on serverless) and once (deprecated)" width="800">

| Output Mode | Description | Use Case |
|------|-------------|----------|
| **Append** | Only new rows | Raw data ingestion (stateless) |
| **Update** | Only updated rows | Aggregations (stateful) |
| **Complete** | Entire result rewritten | Small aggregations |

In [0]:
# Remove checkpoints and schemas from an earlier run, so Auto Loader reads all files again
dbutils.fs.rm(CHECKPOINT_BASE_PATH, True)
dbutils.fs.rm(SCHEMA_BASE_PATH, True)
print("Old checkpoints and schema locations removed")

In [0]:
# Target table, checkpoint and schema location for the Auto Loader demo
TARGET_TABLE_AL = f"{BRONZE_SCHEMA}.orders_autoloader"
CHECKPOINT_AL = f"{CHECKPOINT_BASE_PATH}/autoloader"
SCHEMA_AL = f"{SCHEMA_BASE_PATH}/autoloader"

You configure Auto Loader with `cloudFiles.*` options on `readStream`.

### Auto Loader Configuration Options

| Category | Key Options |
|---|---|
| **Common** | `cloudFiles.format`, `cloudFiles.schemaLocation`, `cloudFiles.includeExistingFiles` |
| **Schema** | `cloudFiles.inferColumnTypes`, `cloudFiles.schemaEvolutionMode` (`addNewColumns`, `rescue`, `failOnNewColumns`, `none`) |
| **Rate limiting** | `cloudFiles.maxFilesPerTrigger`, `cloudFiles.maxBytesPerTrigger` |
| **File detection** | `cloudFiles.useManagedFileEvents` (file events, recommended), `cloudFiles.useNotifications` (legacy notifications: Event Grid/Queue Storage, SNS/SQS, Pub/Sub) |

[Full options reference](https://learn.microsoft.com/en-us/azure/databricks/ingestion/cloud-object-storage/auto-loader/options/)

### File Detection Modes: Directory Listing vs File Notification

Auto Loader must find out which files in the input directory are new. The way it does this matters when the directory holds many files.

| Mode | How new files are discovered | Enable with | Notes |
|---|---|---|---|
| **Directory listing** | Lists the input directory on every micro-batch | nothing; used when file events are not available | Minimal permissions, quick to start; listing cost grows with the number of files |
| **File notification with file events** (recommended) | Reads the file-event cache Unity Catalog keeps for an **external location** that has file events enabled | automatic on serverless and DBR 18.1+ when the location has file events (`cloudFiles.useManagedFileEvents` defaults to `if_available`); `"true"` requires them | No queues or extra cloud permissions in the stream; DBR 14.3 LTS+; scales to millions of files |
| **Legacy file notification** | Subscribes to cloud storage events through a queue (Azure Event Grid + Queue Storage, AWS SNS/SQS, GCP Pub/Sub) | `.option("cloudFiles.useNotifications", "true")` | Needs permissions to create/read the queue; still supported, new workloads should use file events |

> **Exam Tip:** Auto Loader's file discovery modes are **directory listing** (lists the directory, simplest setup) and **file notification** (event-driven, for very large or high-volume directories). On serverless and DBR 18.1+ `cloudFiles.useManagedFileEvents` defaults to `if_available`: Auto Loader reads file events when the storage location has them enabled and falls back to directory listing otherwise. `cloudFiles.useIncrementalListing` is deprecated.

<img src="../../../assets/images/nb04-file-detection.png" alt="How Auto Loader finds new files: directory listing, file events (recommended, default on serverless) and legacy file notification" width="800">

In [0]:
# Drop the target from an earlier run; its checkpoint was removed above
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE_AL}")
print(f"Dropped {TARGET_TABLE_AL} if it existed")

In [0]:
# Define the Auto Loader stream; files are loaded only when writeStream starts it
df_autoloader = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_AL)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(STREAM_SOURCE_PATH) # Reading from our simulated stream source
)

Add the processing time and the source file path to every row. With `_source_file` you can trace a bad row back to the file it came from.

In [0]:
# Processing time and source file path on every row
from pyspark.sql.functions import col

df_enriched = (df_autoloader
    .withColumn("_processing_time", F.current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

Start the stream with the `availableNow` trigger. It processes all available data and then stops, like a batch job, so it fits a scheduled run.

> **Note:** the preview lists rows with `customer_id = null` last. About 3% of the orders have a NULL customer id in the source data.

In [0]:
# One availableNow run: load all files that are in the folder now
query = (df_enriched.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_AL)
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE_AL)
)
query.awaitTermination()   # availableNow stops by itself — wait before reading the table
display(spark.sql(f"SELECT * FROM {TARGET_TABLE_AL} ORDER BY customer_id NULLS LAST"))

The next table shows how many records were loaded and from how many source files.

In [0]:
# Rows loaded and number of distinct source files
display(
    spark.createDataFrame([
        ("Records Loaded", str(spark.table(TARGET_TABLE_AL).count())),
        ("Source Files", str(spark.table(TARGET_TABLE_AL).select("_source_file").distinct().count()))
    ], ["Metric", "Value"])
)

`Source Files` counts files, not batches: Spark writes each batch as a folder of `part-*` files.

### Demo: Incremental Processing: Add New Data

A new batch lands in the source directory. When you run the same stream with the **same checkpoint** again, it reads only the new files. A run with nothing new adds 0 rows, and the output shows the new rows for each run.

<img src="../../../assets/images/nb04-autoloader-runs.png" alt="Three availableNow runs of one Auto Loader stream: new files in the landing folder, the checkpoint and the rows added to the target table" width="800">

In [0]:
# Batch 2 arrives in the landing directory
stream_batches[1].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/batch_02")
print(f"Batch 2 written to {STREAM_SOURCE_PATH}/batch_02")

In [0]:
# One scheduled run with the same stream and checkpoint: only new files are read
def run_autoloader_once(label):
    """One scheduled incremental run: same stream definition + same checkpoint, availableNow trigger."""
    before = spark.table(TARGET_TABLE_AL).count()
    q = (df_enriched.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_AL)
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE_AL)
    )
    q.awaitTermination()
    after = spark.table(TARGET_TABLE_AL).count()
    print(f"{label:<32} +{after - before:>6,} new rows   (total {after:,})")
    return after - before

run_autoloader_once("Run after batch_02 arrived")
added_on_rerun = run_autoloader_once("Run again, nothing new")

### Demo: Repeated Scheduled Runs (`availableNow`)

In production an ingestion job usually runs on a schedule, for example every 15 minutes, with `trigger(availableNow=True)`. Each run processes what arrived since the last checkpoint and then stops, so no compute runs between the runs. In the next cell three more batches arrive one by one, and a run follows each arrival. The table after it counts rows per batch folder, and each batch appears once.

> **Classic compute only:** a continuously running stream uses a time-based trigger, e.g. `.trigger(processingTime="10 seconds")`, and runs until `query.stop()`. Serverless compute supports only `availableNow` and the deprecated `once`. A `processingTime` trigger raises an error there. The next demo shows a running stream and what the same code does on serverless.

In [0]:
# Simulate arrival of batches 3-5, with one scheduled run after each arrival
for i in range(2, 5):
    batch_num = i + 1
    stream_batches[i].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/batch_{batch_num:02d}")
    run_autoloader_once(f"Run after batch_{batch_num:02d} arrived")

In [0]:
# Rows per source batch folder: each batch is loaded exactly once
display(
    spark.table(TARGET_TABLE_AL)
    .withColumn("batch_dir", F.regexp_extract("_source_file", r"(batch_\d+)", 1))
    .groupBy("batch_dir").count()
    .orderBy("batch_dir")
)

### Demo: A Running Stream (`processingTime`)

Use a time-based trigger when new data must reach the table within seconds. Such a stream keeps running and checks the source every few seconds. In the next cells batches 6–10 land in the source folder one by one, a few seconds apart, and the row count of the target grows after each arrival.

- On classic compute the query starts with `trigger(processingTime="5 seconds")` and runs until `stop()`.
- On serverless the same start fails with `INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED`. The cell catches the error and runs one `availableNow` pass after each arrival instead, so the row count grows the same way. For an always-on stream on serverless, use a Lakeflow pipeline in continuous mode (module 07).

The stream has its own target table, checkpoint and schema location, so it starts from the beginning of the source folder (batches 1–5 included).

In [ ]:
# Separate table, checkpoint and schema location, so this stream starts from the first file
import time

LIVE_TABLE = f"{TARGET_TABLE_AL}_live"
LIVE_CHECKPOINT = f"{CHECKPOINT_BASE_PATH}/autoloader_live"
spark.sql(f"DROP TABLE IF EXISTS {LIVE_TABLE}")
dbutils.fs.rm(LIVE_CHECKPOINT, True)

df_live = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{SCHEMA_BASE_PATH}/autoloader_live")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(STREAM_SOURCE_PATH)
    .withColumn("_source_file", col("_metadata.file_path"))
)

def live_writer():
    return (df_live.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", LIVE_CHECKPOINT))

def live_count():
    return spark.table(LIVE_TABLE).count() if spark.catalog.tableExists(LIVE_TABLE) else 0

# Try a running stream; serverless rejects time-based triggers
try:
    live_query = live_writer().trigger(processingTime="5 seconds").toTable(LIVE_TABLE)
    time.sleep(5)
    if not live_query.isActive and live_query.exception():
        raise Exception(str(live_query.exception()))
    print("Running stream started: processingTime = 5 seconds (classic compute)")
except Exception as e:
    if "INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED" not in str(e):
        raise
    live_query = None
    print("Serverless: INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED for processingTime, "
          "so one availableNow pass runs after each arrival")

In [ ]:
# Batches 6-10 arrive a few seconds apart; watch the row count grow
for i in range(5, 10):
    batch = f"batch_{i + 1:02d}"
    stream_batches[i].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/{batch}")
    if live_query is None:
        live_writer().trigger(availableNow=True).toTable(LIVE_TABLE).awaitTermination()
    else:
        time.sleep(10)  # the running stream picks the files up on its next trigger
    print(f"{batch} arrived -> {live_count():>7,} rows in {LIVE_TABLE}")

In [ ]:
# Stop the running stream (classic compute); on serverless nothing is running
if live_query is not None:
    live_query.stop()
print(f"{LIVE_TABLE}: {live_count():,} rows, no stream running")

## Error Handling

Real source files sometimes contain malformed records, corrupted files or values that do not match the schema. The read mode decides what happens to such records during ingestion.

| Mode | Behavior |
|------|----------|
| `PERMISSIVE` | Parses what it can, errors → `_corrupt_record` |
| `DROPMALFORMED` | Removes malformed records |
| `FAILFAST` | Stops on first error |

> **Best Practice:** Use `badRecordsPath` to save malformed records for later analysis.
>
> This section is reference only. The runnable `badRecordsPath` demo is in <a href="$./BONUS_04_streaming_advanced">BONUS_04 — Streaming Advanced</a>. Below is the Auto Loader alternative: rescue mode.



### Schema Evolution & Rescued Data

Source systems add or change columns without warning. `schemaEvolutionMode` decides what Auto Loader does when the data no longer matches the schema.

| `schemaEvolutionMode` | Behavior |
|---|---|
| `addNewColumns` | The stream stops on a new column, records it in the schema location and picks it up after a restart (a job retry does that); **default when no schema is provided** |
| `rescue` | New/mismatched → `_rescued_data` JSON column |
| `failOnNewColumns` | Fail if schema changes |
| `none` | Ignores new columns, **default when a schema is provided** via `.schema(...)` |

> **Exam Tip:** the `_rescued_data` column captures data that does not fit the schema: columns missing from the schema, type mismatches and case mismatches. It does not capture malformed or corrupt records. Those depend on the parser mode (`PERMISSIVE`, `DROPMALFORMED`, `FAILFAST`) or go to `badRecordsPath`. With an explicit `.schema(...)` the default mode is `none`, so you must set `rescue` explicitly (as below).

<img src="../../../assets/images/nb04-schema-evolution-modes.png" alt="A new column arrives: what Auto Loader does in addNewColumns, rescue, failOnNewColumns and none mode" width="800">

In [0]:
# Target table, checkpoint and schema location for the rescue demo
TARGET_TABLE_RESCUE = f"{BRONZE_SCHEMA}.orders_rescued"
CHECKPOINT_RESCUE = f"{CHECKPOINT_BASE_PATH}/rescue"
SCHEMA_RESCUE = f"{SCHEMA_BASE_PATH}/rescue"

spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE_RESCUE}")
print(f"Dropped {TARGET_TABLE_RESCUE} if it existed")

An explicit schema keeps the column types under your control. This one has only three columns on purpose, so you can see where the other fields go.

In [0]:
# Only 3 columns on purpose, so the other fields go to _rescued_data
partial_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("total_amount", DoubleType(), True)
])

Write one bad record with an extra column and a text value in a number column. Then read the folder with Auto Loader in rescue mode.

In [0]:
# One bad record: an extra column and a text value in total_amount
bad_data = [{"order_id": 99999, "total_amount": "INVALID_NUMBER", "new_col": "surprise"}]
spark.createDataFrame(bad_data).write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/bad_data")

In [0]:
# Rescue mode with a partial schema: fields outside the schema go to _rescued_data
df_rescue = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_RESCUE)
    .option("cloudFiles.schemaEvolutionMode", "rescue")  # Rescue mode!
    .schema(partial_schema)  # Partial schema
    .load(STREAM_SOURCE_PATH)
)

Run the stream once with `availableNow` and show the target table.

In [0]:
# One availableNow run, then show the table
query_rescue = (df_rescue.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_RESCUE)
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE_RESCUE)
)
query_rescue.awaitTermination()
display(spark.sql(f"SELECT * FROM {TARGET_TABLE_RESCUE} ORDER BY customer_id NULLS LAST"))

Print the schema: the `_rescued_data` column appears next to the three columns you defined.

In [0]:
# _rescued_data appears next to the three schema columns
spark.table(TARGET_TABLE_RESCUE).printSchema()

Find the bad record in `_rescued_data`.

> With the partial schema *every* row has `_rescued_data`, because columns like `product_id` are not in the schema. Filter on `new_col` to see the injected record. The unexpected column and the `"INVALID_NUMBER"` value are kept as JSON, and `total_amount` is `null` because of the type mismatch. Auto Loader also adds the source file path to the rescued JSON under the key `_file_path`.

In [0]:
# Show only the injected bad record
display(
    spark.table(TARGET_TABLE_RESCUE)
    .filter(F.col("_rescued_data").contains("new_col"))   # the injected bad record
)

## Lakeflow Connect (Informational)

> **Self-study:** skipped in class; not required by any lab. Lakeflow Connect is shown live in `04a_lakeflow_connect_demo`.

Lakeflow Connect gives zero-code ingestion from SaaS systems such as Salesforce, Workday, HubSpot and ServiceNow, set up in the UI. SAP is not a Lakeflow Connect source: SAP Business Data Cloud shares its data with Databricks through the SAP BDC connector, which uses Delta Sharing (OpenSharing). The table shows which tool fits which source.

| Method | Use Case |
|--------|----------|
| **COPY INTO** | Files in cloud storage (batch) |
| **Auto Loader** | Files in cloud storage (streaming) |
| **Lakeflow Connect** | Data from SaaS systems |
| **Lakeflow Spark Declarative Pipelines** | Transformations Bronze → Silver → Gold |



## Summary

| Topic | Key Concept | Exam Keywords |
|---|---|---|
| **COPY INTO** | Idempotent batch loading, file tracking | `COPY INTO`, `FILEFORMAT`, `COPY_OPTIONS ('mergeSchema' = 'true')`, `'force' = 'true'` |
| **Auto Loader** | `cloudFiles` streaming ingestion | `cloudFiles.format`, `schemaLocation`, `schemaEvolutionMode` |
| **File detection modes** | Directory listing vs file notification; file events are used when available (serverless, DBR 18.1+) | `cloudFiles.useManagedFileEvents`, `cloudFiles.useNotifications` |
| **Trigger Modes** | `availableNow=True` (serverless + classic), `processingTime` (classic only) | Scheduled incremental vs continuous |
| **Schema Evolution** | `rescue`, `addNewColumns`, `failOnNewColumns` | `_rescued_data` column |
| **Lakeflow Connect** | Zero-code SaaS ingestion | Salesforce, Workday, ServiceNow |

> Stream-static joins, watermarking, `badRecordsPath` and CDF for incremental ETL: <a href="$./BONUS_04_streaming_advanced">BONUS_04 — Streaming Advanced</a>

## Cleanup

There is nothing to run here. The tables of this demo stay in your catalog, because `BONUS_04` reuses `customers_batch`, and the catalog is removed after the training. To drop them now, run `DROP TABLE IF EXISTS <CATALOG>.bronze.<table>` for `customers_batch`, `orders_autoloader`, `orders_autoloader_live` and `orders_rescued`, then `dbutils.fs.rm(CHECKPOINT_BASE_PATH, True)`.

← <a href="$./03_delta_lake">03 — Delta Lake</a> | **<a href="$../../..">Course folder (README)</a>** | <a href="$../../day2/demo/04a_lakeflow_connect_demo">04a — Lakeflow Connect (Day 2) →</a>

## Self-check (trainer): participants can skip this cell

In [0]:
# Self-check (trainer): the incremental loads added each file once (no stream runs again here)

# COPY INTO: the second run on the same files added 0 rows; loading days 2-4 added rows
assert count_before > 0 and count_after == count_before, (count_before, count_after)
assert count_after_days > count_before_days, (count_before_days, count_after_days)

# Auto Loader: the scheduled run with nothing new added 0 rows; the live stream table got rows
assert added_on_rerun == 0, added_on_rerun
assert spark.table(LIVE_TABLE).count() > 0, "the live stream wrote no rows"

# Rescue mode: the bad record kept its fields in _rescued_data
rescued = spark.table(TARGET_TABLE_RESCUE)
assert rescued.filter(F.col("_rescued_data").isNotNull()).count() > 0, "no row in _rescued_data"

print("Self-check passed")